# NB03：灰分與鈉離子（Ash Content and Sodium Ion Determination）

**食品分析實驗 週次 3｜Nielsen's Food Analysis, 6th ed. 對應章節：灰分分析（Ash）、礦物質分析（Minerals）**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 由坩堝空重、坩堝+樣品重、灰化後坩堝+灰分重，計算**灰分 %**，並比較黑糖 vs 白糖、糙米 vs 白米。
2. 由 Na⁺ ISE 讀數與**稀釋倍數**回推樣品的鈉濃度，換算成 **NaCl 當量**（× 2.542 = 58.44/22.99）。
3. 計算**標準添加回收率 %**，用 90–110% 的教學判準評估方法準確性。
4. 檢查**飄移(drift)**：每測 10 個樣品重測一次標準液，抓出偏差 > ±5% 的情況。
5. 比較量測鈉 vs 營養標示鈉（換算為每 100 g），並討論非 NaCl 的含鈉來源。
6. （選用）由兩個標準液的 mV 讀數計算 Nernst 響應斜率，跟理論值 59.2 mV/decade 比較。

## 如何使用本筆記本

用法與 NB01/NB02 相同：直接「全部執行」使用內建模擬資料；`load_data(source)` 支援本機路徑、已發布 Google 試算表 CSV 連結、Colab 上傳，或退回內嵌字串。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。

## 資料格式 (Data Schema)

沿用共用長格式，欄位同 NB02（含 `sample`）：`group, student, item, rep, value, unit, temp_C, note, sample`。本週的 `item` 值包括：`crucible_empty_g`、`crucible_sample_g`、`crucible_ash_g`（灰分）；`na_ise_ppm`（Na⁺讀數）；`std_addition_before_ppm`/`std_addition_after_ppm`（標準添加回收）；`drift_check_ppm`（飄移監控，`group=0` 代表 QC 標準液，非特定組別）；`nernst_mV`（兩點 Nernst 響應）。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note,sample
1,G1_S1,crucible_empty_g,1,28.6437,g,555.0,,black_sugar
1,G1_S1,crucible_sample_g,1,31.0917,g,555.0,,black_sugar
1,G1_S1,crucible_ash_g,1,28.6815,g,555.0,,black_sugar
1,G1_S1,crucible_empty_g,1,24.5651,g,548.0,,white_sugar
1,G1_S1,crucible_sample_g,1,27.0,g,548.0,,white_sugar
1,G1_S1,crucible_ash_g,1,24.5659,g,548.0,,white_sugar
1,G1_S1,crucible_empty_g,1,24.7687,g,554.0,,brown_rice
1,G1_S1,crucible_sample_g,1,27.226,g,554.0,,brown_rice
1,G1_S1,crucible_ash_g,1,24.8,g,554.0,,brown_rice
1,G1_S1,crucible_empty_g,1,27.8632,g,546.0,,polished_rice
1,G1_S1,crucible_sample_g,1,30.4196,g,546.0,,polished_rice
1,G1_S1,crucible_ash_g,1,27.8765,g,546.0,,polished_rice
1,G1_S1,crucible_empty_g,1,27.3275,g,550.0,,milk_powder
1,G1_S1,crucible_sample_g,1,29.7796,g,550.0,,milk_powder
1,G1_S1,crucible_ash_g,1,27.4826,g,550.0,,milk_powder
2,G2_S1,crucible_empty_g,1,28.5485,g,549.0,,black_sugar
2,G2_S1,crucible_sample_g,1,31.0145,g,549.0,,black_sugar
2,G2_S1,crucible_ash_g,1,28.5877,g,549.0,,black_sugar
2,G2_S1,crucible_empty_g,1,28.6703,g,552.0,,white_sugar
2,G2_S1,crucible_sample_g,1,31.1527,g,552.0,,white_sugar
2,G2_S1,crucible_ash_g,1,28.6712,g,552.0,,white_sugar
2,G2_S1,crucible_empty_g,1,24.9257,g,548.0,,brown_rice
2,G2_S1,crucible_sample_g,1,27.4472,g,548.0,,brown_rice
2,G2_S1,crucible_ash_g,1,24.9604,g,548.0,,brown_rice
2,G2_S1,crucible_empty_g,1,25.955,g,556.0,,polished_rice
2,G2_S1,crucible_sample_g,1,28.4143,g,556.0,,polished_rice
2,G2_S1,crucible_ash_g,1,25.9679,g,556.0,,polished_rice
2,G2_S1,crucible_empty_g,1,24.7795,g,553.0,,milk_powder
2,G2_S1,crucible_sample_g,1,27.2375,g,553.0,,milk_powder
2,G2_S1,crucible_ash_g,1,24.9195,g,553.0,,milk_powder
3,G3_S1,crucible_empty_g,1,26.6229,g,551.0,,black_sugar
3,G3_S1,crucible_sample_g,1,29.1501,g,551.0,,black_sugar
3,G3_S1,crucible_ash_g,1,26.6584,g,551.0,,black_sugar
3,G3_S1,crucible_empty_g,1,26.3249,g,551.0,,white_sugar
3,G3_S1,crucible_sample_g,1,28.8358,g,551.0,,white_sugar
3,G3_S1,crucible_ash_g,1,26.3259,g,551.0,,white_sugar
3,G3_S1,crucible_empty_g,1,25.1994,g,553.0,,brown_rice
3,G3_S1,crucible_sample_g,1,27.7028,g,553.0,,brown_rice
3,G3_S1,crucible_ash_g,1,25.23,g,553.0,,brown_rice
3,G3_S1,crucible_empty_g,1,28.231,g,547.0,,polished_rice
3,G3_S1,crucible_sample_g,1,30.715,g,547.0,,polished_rice
3,G3_S1,crucible_ash_g,1,28.243,g,547.0,,polished_rice
3,G3_S1,crucible_empty_g,1,24.8388,g,555.0,,milk_powder
3,G3_S1,crucible_sample_g,1,27.4135,g,555.0,,milk_powder
3,G3_S1,crucible_ash_g,1,24.985,g,555.0,,milk_powder
4,G4_S1,crucible_empty_g,1,27.3914,g,553.0,,black_sugar
4,G4_S1,crucible_sample_g,1,29.8747,g,553.0,,black_sugar
4,G4_S1,crucible_ash_g,1,27.4283,g,553.0,,black_sugar
4,G4_S1,crucible_empty_g,1,27.3552,g,548.0,,white_sugar
4,G4_S1,crucible_sample_g,1,29.8949,g,548.0,,white_sugar
4,G4_S1,crucible_ash_g,1,27.3559,g,548.0,,white_sugar
4,G4_S1,crucible_empty_g,1,25.2875,g,544.0,,brown_rice
4,G4_S1,crucible_sample_g,1,27.7779,g,544.0,,brown_rice
4,G4_S1,crucible_ash_g,1,25.3148,g,544.0,,brown_rice
4,G4_S1,crucible_empty_g,1,24.3498,g,553.0,,polished_rice
4,G4_S1,crucible_sample_g,1,26.8747,g,553.0,,polished_rice
4,G4_S1,crucible_ash_g,1,24.3626,g,553.0,,polished_rice
4,G4_S1,crucible_empty_g,1,27.3422,g,548.0,,milk_powder
4,G4_S1,crucible_sample_g,1,29.8501,g,548.0,,milk_powder
4,G4_S1,crucible_ash_g,1,27.4985,g,548.0,,milk_powder
5,G5_S1,crucible_empty_g,1,28.8841,g,548.0,,black_sugar
5,G5_S1,crucible_sample_g,1,31.351,g,548.0,,black_sugar
5,G5_S1,crucible_ash_g,1,28.9195,g,548.0,,black_sugar
5,G5_S1,crucible_empty_g,1,28.3342,g,550.0,,white_sugar
5,G5_S1,crucible_sample_g,1,30.8585,g,550.0,,white_sugar
5,G5_S1,crucible_ash_g,1,28.3348,g,550.0,,white_sugar
5,G5_S1,crucible_empty_g,1,24.9139,g,550.0,,brown_rice
5,G5_S1,crucible_sample_g,1,27.4362,g,550.0,,brown_rice
5,G5_S1,crucible_ash_g,1,24.9455,g,550.0,,brown_rice
5,G5_S1,crucible_empty_g,1,25.8091,g,543.0,,polished_rice
5,G5_S1,crucible_sample_g,1,28.3051,g,543.0,,polished_rice
5,G5_S1,crucible_ash_g,1,25.8199,g,543.0,,polished_rice
5,G5_S1,crucible_empty_g,1,24.708,g,545.0,,milk_powder
5,G5_S1,crucible_sample_g,1,27.1581,g,545.0,,milk_powder
5,G5_S1,crucible_ash_g,1,24.8586,g,545.0,,milk_powder
1,G1_S1,na_ise_ppm,1,1198.4,ppm,25.9,,soy_sauce
1,G1_S1,na_ise_ppm,1,273.9,ppm,25.4,,potato_chips
1,G1_S1,na_ise_ppm,1,153.2,ppm,24.5,,instant_soup
1,G1_S1,na_ise_ppm,1,200.4,ppm,25.8,,sports_drink
1,G1_S1,na_ise_ppm,1,232.0,ppm,24.4,,milk_powder
2,G2_S1,na_ise_ppm,1,1216.8,ppm,25.2,,soy_sauce
2,G2_S1,na_ise_ppm,1,272.4,ppm,25.7,,potato_chips
2,G2_S1,na_ise_ppm,1,156.2,ppm,25.4,,instant_soup
2,G2_S1,na_ise_ppm,1,205.6,ppm,25.3,,sports_drink
2,G2_S1,na_ise_ppm,1,235.7,ppm,25.3,,milk_powder
3,G3_S1,na_ise_ppm,1,1183.8,ppm,24.8,,soy_sauce
3,G3_S1,na_ise_ppm,1,277.3,ppm,25.0,,potato_chips
3,G3_S1,na_ise_ppm,1,158.5,ppm,24.3,,instant_soup
3,G3_S1,na_ise_ppm,1,211.0,ppm,25.2,,sports_drink
3,G3_S1,na_ise_ppm,1,224.9,ppm,25.9,,milk_powder
4,G4_S1,na_ise_ppm,1,1198.2,ppm,24.7,,soy_sauce
4,G4_S1,na_ise_ppm,1,279.8,ppm,24.0,,potato_chips
4,G4_S1,na_ise_ppm,1,151.6,ppm,25.0,,instant_soup
4,G4_S1,na_ise_ppm,1,206.0,ppm,24.2,,sports_drink
4,G4_S1,na_ise_ppm,1,237.6,ppm,25.0,,milk_powder
5,G5_S1,na_ise_ppm,1,1216.4,ppm,25.1,,soy_sauce
5,G5_S1,na_ise_ppm,1,263.2,ppm,24.5,,potato_chips
5,G5_S1,na_ise_ppm,1,153.6,ppm,25.0,,instant_soup
5,G5_S1,na_ise_ppm,1,201.9,ppm,24.0,,sports_drink
5,G5_S1,na_ise_ppm,1,233.1,ppm,25.8,,milk_powder
1,G1_S2,std_addition_before_ppm,1,297.6,ppm,25.0,,soy_sauce
1,G1_S2,std_addition_after_ppm,1,497.8,ppm,25.0,,soy_sauce
2,G2_S2,std_addition_before_ppm,1,274.9,ppm,25.0,,soy_sauce
2,G2_S2,std_addition_after_ppm,1,477.7,ppm,25.0,,soy_sauce
3,G3_S2,std_addition_before_ppm,1,299.2,ppm,25.0,,soy_sauce
3,G3_S2,std_addition_after_ppm,1,503.2,ppm,25.0,,soy_sauce
4,G4_S2,std_addition_before_ppm,1,302.0,ppm,25.0,,soy_sauce
4,G4_S2,std_addition_after_ppm,1,453.7,ppm,25.0,回收率 < 90%，疑似基質效應或加標體積誤差，建議重做,soy_sauce
5,G5_S2,std_addition_before_ppm,1,292.5,ppm,25.0,,soy_sauce
5,G5_S2,std_addition_after_ppm,1,485.6,ppm,25.0,,soy_sauce
0,QC_standard,drift_check_ppm,0,988.4,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,10,991.3,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,20,1025.5,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,30,990.1,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,40,1061.0,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,50,989.2,ppm,25.0,,Na_1000ppm_std
1,G1_S3,nernst_mV,1,101.1,mV,25.0,,std_100ppm
1,G1_S3,nernst_mV,1,159.7,mV,25.0,,std_1000ppm
2,G2_S3,nernst_mV,1,99.8,mV,25.0,,std_100ppm
2,G2_S3,nernst_mV,1,159.2,mV,25.0,,std_1000ppm
3,G3_S3,nernst_mV,1,99.2,mV,25.0,,std_100ppm
3,G3_S3,nernst_mV,1,159.7,mV,25.0,,std_1000ppm
4,G4_S3,nernst_mV,1,99.5,mV,25.0,,std_100ppm
4,G4_S3,nernst_mV,1,157.7,mV,25.0,,std_1000ppm
5,G5_S3,nernst_mV,1,98.5,mV,25.0,,std_100ppm
5,G5_S3,nernst_mV,1,159.4,mV,25.0,,std_1000ppm
"""

In [ ]:
def load_data(source=None):
    """Load NB03 ash/sodium data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note, sample
    (see NB01/NB02 for the load_data() source-resolution rules)."""
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb03_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['item'].nunique()} 個量測項目、{df['sample'].nunique()} 種樣品。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 灰分 % (Ash Content)

$$灰分\% = \frac{灰分重}{樣品重} \times 100$$

### 範例（沿用課程講義數字，黑糖）

坩堝空重 W₀ = **26.184 g**；坩堝＋樣品 = 28.696 g → 樣品重 = **2.512 g**；灰化後坩堝＋灰分(恆重) = 26.221 g → 灰分重 = **0.037 g**；灰分 % = 0.037 ÷ 2.512 × 100 ≈ **1.47%**。

In [ ]:
W0, W_sample_total, W_ash_total = 26.184, 28.696, 26.221
sample_mass = W_sample_total - W0
ash_mass = W_ash_total - W0
ash_pct_example = ash_mass / sample_mass * 100
print(f"樣品重 = {sample_mass:.3f} g, 灰分重 = {ash_mass:.3f} g, 灰分% = {ash_pct_example:.2f}%  (expect ~1.47%)")

In [ ]:
ash_items = ["crucible_empty_g", "crucible_sample_g", "crucible_ash_g"]
ash_wide = (
    df[df["item"].isin(ash_items)]
    .pivot_table(index=["group", "sample"], columns="item", values="value", aggfunc="first")
    .reset_index()
)
ash_wide["sample_mass_g"] = ash_wide["crucible_sample_g"] - ash_wide["crucible_empty_g"]
ash_wide["ash_mass_g"] = ash_wide["crucible_ash_g"] - ash_wide["crucible_empty_g"]
ash_wide["ash_pct"] = ash_wide["ash_mass_g"] / ash_wide["sample_mass_g"] * 100
ash_wide[["group", "sample", "sample_mass_g", "ash_mass_g", "ash_pct"]]

In [ ]:
ash_summary = ash_wide.groupby("sample")["ash_pct"].agg(n="count", mean_pct="mean", sd_pct="std").reset_index()
ash_summary["rsd_pct"] = ash_summary["sd_pct"] / ash_summary["mean_pct"] * 100
ash_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = ash_summary.sort_values("mean_pct")["sample"]
ax.bar(order, ash_summary.set_index("sample").loc[order, "mean_pct"],
       yerr=ash_summary.set_index("sample").loc[order, "sd_pct"], capsize=4, color="tab:brown", alpha=0.85)
ax.set_ylabel("Ash content (%)")
ax.set_title("Ash % by sample (black vs white sugar, brown vs polished rice, blind sample)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

黑糖/糙米的灰分明顯高於白糖/白米——這是**精製程度**的指標之一，但灰分本身**不能**告訴你礦物質的種類或鈉/鹽含量，只能告訴你「總無機殘留物」的比例（延續第 3–4 節，用 Na⁺ ISE 才能回答「鈉」的問題）。全脂奶粉是本課程共同盲樣，此處的灰分結果會延續到後續週次的品管追蹤。

## 3. Na⁺ ISE 讀數、稀釋倍數、NaCl 當量換算

$$C_{樣品} = 讀數 \times 稀釋倍數 \qquad NaCl\ 當量 = Na \times 2.542\ \left(=\frac{58.44}{22.99}\right)$$

### 範例（沿用課程講義數字）

醬油以去離子水稀釋 **1:50** 後，用 Na⁺ ISE 測得讀數 **1100 ppm(mg/L)**：

In [ ]:
NACL_FACTOR = 58.44 / 22.99
reading_ppm, dilution = 1100.0, 50
C_Na_gL = reading_ppm * dilution / 1000.0   # g/L in the original soy sauce
C_Na_100mL = C_Na_gL / 10.0                  # g/100 mL
C_NaCl_100mL = C_Na_100mL * NACL_FACTOR
print(f"Na 濃度 = {C_Na_100mL:.2f} g/100 mL")
print(f"NaCl 當量 = {C_NaCl_100mL:.2f} g/100 mL  (expect ~13.98)")

In [ ]:
na_df = df[df["item"] == "na_ise_ppm"][["group", "sample", "value"]].rename(columns={"value": "reading_ppm"})
na_df

## 4. 標準添加回收率 (Standard-Addition Recovery)

$$回收率\% = \frac{C_{加標} - C_{未加標}}{C_{加標量}} \times 100$$

教學上常以 **90–110%** 作為可接受範圍。

In [ ]:
STD_ADD_PPM = 200.0
before = df[df["item"] == "std_addition_before_ppm"][["group", "value"]].rename(columns={"value": "c_before"})
after = df[df["item"] == "std_addition_after_ppm"][["group", "value", "note"]].rename(columns={"value": "c_after"})
recovery = before.merge(after, on="group")
recovery["recovery_pct"] = (recovery["c_after"] - recovery["c_before"]) / STD_ADD_PPM * 100
recovery["pass_90_110"] = recovery["recovery_pct"].between(90, 110)
recovery

## 5. 飄移檢查 (Drift Check)

每量測約 10 個樣品，重新測一次已知濃度的標準液（此處標準液標稱濃度 1000 ppm），偏差超過 **±5%** 應重新校正並回頭檢視受影響的樣品。

In [ ]:
drift = df[df["item"] == "drift_check_ppm"][["rep", "value"]].rename(columns={"rep": "sample_index", "value": "reading_ppm"})
NOMINAL_STD_PPM = 1000.0
drift["deviation_pct"] = (drift["reading_ppm"] - NOMINAL_STD_PPM) / NOMINAL_STD_PPM * 100
drift["flagged"] = drift["deviation_pct"].abs() > 5
drift

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(drift["sample_index"], drift["deviation_pct"], "o-", color="tab:blue")
ax.axhspan(-5, 5, color="tab:green", alpha=0.12, label="acceptable (+/-5%)")
flagged = drift[drift["flagged"]]
if len(flagged) > 0:
    ax.scatter(flagged["sample_index"], flagged["deviation_pct"], s=120, facecolors="none",
               edgecolors="tab:red", linewidths=2, label="flagged: drift > 5%", zorder=5)
ax.axhline(0, color="grey", linewidth=1)
ax.set_xlabel("Sample index (standard re-measured every 10 samples)")
ax.set_ylabel("Deviation from nominal 1000 ppm (%)")
ax.set_title("Drift check across the measurement run")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 量測鈉 vs 營養標示鈉

把量測結果換算成「每 100 g」跟包裝上「每份」的標示鈉互相比較（先把標示的 mg/份 除以每份克數再乘 100 換算成同一基礎）。差異可能來自：秤重/稀釋誤差、批次差異，以及**非 NaCl 的含鈉來源**（味精 MSG、小蘇打 NaHCO₃、苯甲酸鈉等）——這些都貢獻鈉，但不是鹽。

In [ ]:
NA_PRODUCTS = {
    "soy_sauce":     {"dilution": 50, "label_mg_serving": 850, "serving_g": 15},
    "potato_chips":  {"dilution": 1,  "label_mg_serving": 180, "serving_g": 30},
    "instant_soup":  {"dilution": 20, "label_mg_serving": 750, "serving_g": 250},
    "sports_drink":  {"dilution": 1,  "label_mg_serving": 110, "serving_g": 500},
    "milk_powder":   {"dilution": 20, "label_mg_serving": 120, "serving_g": 25},
}

rows = []
for product, cfg in NA_PRODUCTS.items():
    sub = na_df[na_df["sample"] == product]
    mean_reading = sub["reading_ppm"].mean()
    # potato chips: 5 g sample extracted into 100 mL, measured on the
    # extract directly (dilution factor folded into the 100/5 mL:g ratio)
    if product == "potato_chips":
        measured_mg_100g = mean_reading * 100.0 / 1000.0 * (100.0 / 5.0)
    else:
        measured_mg_100g = mean_reading * cfg["dilution"] / 1000.0 * 100.0
    label_mg_100g = cfg["label_mg_serving"] / cfg["serving_g"] * 100.0
    rows.append([product, label_mg_100g, measured_mg_100g])

compare_df = pd.DataFrame(rows, columns=["product", "label_mg_100g", "measured_mg_100g"])
compare_df["diff_pct"] = (compare_df["measured_mg_100g"] - compare_df["label_mg_100g"]) / compare_df["label_mg_100g"] * 100
compare_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(compare_df))
width = 0.35
ax.bar(x - width / 2, compare_df["label_mg_100g"], width, label="label (mg/100g)", color="tab:blue")
ax.bar(x + width / 2, compare_df["measured_mg_100g"], width, label="measured (mg/100g)", color="tab:orange")
ax.set_xticks(x)
ax.set_xticklabels(compare_df["product"], rotation=20)
ax.set_ylabel("Sodium (mg / 100 g)")
ax.set_title("Measured vs label sodium")
ax.legend()
plt.tight_layout()
plt.show()

## 7. （選用）簡易 Nernst 斜率檢查

$$slope = \frac{mV_2 - mV_1}{\log_{10}C_2 - \log_{10}C_1}$$

理論值於 25°C 約為 **59.2 mV/decade**。若電極健康狀況不佳，實測斜率會明顯偏離理論值。

In [ ]:
nernst = df[df["item"] == "nernst_mV"][["group", "sample", "value"]].rename(columns={"value": "mV"})
nernst["conc_ppm"] = nernst["sample"].str.extract(r"(\d+)").astype(float)
nernst_wide = nernst.pivot_table(index="group", columns="conc_ppm", values="mV").reset_index()
nernst_wide.columns = ["group", "mV_100ppm", "mV_1000ppm"]
nernst_wide["slope_mV_per_decade"] = (
    (nernst_wide["mV_1000ppm"] - nernst_wide["mV_100ppm"])
    / (np.log10(1000.0) - np.log10(100.0))
)
nernst_wide

In [ ]:
mean_slope = nernst_wide["slope_mV_per_decade"].mean()
print(f"班級平均斜率 = {mean_slope:.1f} mV/decade（理論值 59.2 mV/decade，25°C）")

## 8. 用資料庫檢查實測值：我的數據合理嗎？(Food-composition database check)

第 6 節已經拿量測鈉跟**包裝營養標示**比過。這一節換一個參考點：**官方食物成分資料庫**。要先分清楚三種數字：

| 數字 | 代表什麼 | 常見落差來源 |
|---|---|---|
| **資料庫值** (database) | 某幾個產品/批次，在某一年由某實驗室分析（或計算）出來的平均值，附 n、SD 或 min–max | 品牌、配方、品種、年份不同；有些值是計算/推估，不是分析值 |
| **營養標示** (label) | 廠商依法規標示的每份數值，常是配方計算或定期送驗的結果，會四捨五入 | 與你手上這一包的批次不同；標示基準是「每份」或「每 100 mL」 |
| **實驗值** (lab) | 本班用本班的方法，量「這一包」得到的值 | 方法偏差、單位/稀釋/基準換算錯誤、操作誤差 |

**資料庫 ≠ 標示 ≠ 實驗值**。三者不一致時，先排除「自己算錯」（單位、稀釋倍數、乾/濕基、換算係數），再考慮「方法本身的偏差」，最後才是「樣品本來就不一樣」。下面的小工具會自動幫你檢查前兩類，並提醒第三類。

**資料來源與授權（使用本節數字時請一併註明）**

- **USDA FoodData Central**：SR Legacy（2018-04 最終版）與 Foundation Foods（2026-04-30 版），美國農業部農業研究署（USDA ARS），公共領域／CC0 1.0。<https://fdc.nal.usda.gov>
- **衛生福利部食品藥物管理署「食品營養成分資料庫」**（政府資料開放平臺 [data.gov.tw 資料集 8543](https://data.gov.tw/dataset/8543)「食品營養成分資料集」），依**政府資料開放授權條款第1版**釋出，使用時須註明出處。
- 整理後的檔案：`data/reference_foods.json`（每筆都有資料庫、版本、品項 id、樣本數 n、SD 或 min/max）。由教師端的 `build_reference_foods.py` 直接從官方檔案產生，**沒有任何一個數字是手打或教學假設值**；唯一的教學假設是下面說明的「預設容許差」。

**合理範圍怎麼來的（教學假設要講清楚）**

1. 每一筆資料庫數值先算自己的區間：有 **min–max**（USDA 部分品項）就用 min–max；有 **SD 且 n ≥ 2**（TFND）就用 **平均 ± 2 SD**。
2. 資料庫只代表「被分析的那幾個產品」，不同品牌/批次之間的差異通常更大，所以每筆區間**至少**保留「教學預設容許差」：數值 ± max(相對容許差, 絕對下限)——水分/蛋白質/碳水 ±10%、脂肪/灰分/總糖/鈉 ±15%（絕對下限：水分/蛋白質/脂肪 0.3 g、灰分 0.05 g、碳水 1 g、總糖 0.5 g、鈉 5 mg，皆為每 100 g）。**這是教學假設，不是資料庫數值。**
3. 把各筆區間聯集起來＝「合理範圍」。`within_entries` 欄位另外列出**每一個**資料庫品項是否涵蓋你的數值，避免「只被其中一個資料庫撐住的範圍內」。

In [ ]:
# ---- 內嵌的參考資料子集（離線備援；由 data/reference_foods.json 自動產生，勿手改）----
# 來源與授權見 meta.sources：USDA FoodData Central (CC0) 與 衛福部食藥署 食品營養成分資料庫
# (政府資料開放授權條款第1版，資料集 data.gov.tw 8543)。
REFERENCE_FALLBACK_JSON = r"""{"meta":{"built":"2026-09-27","units":"per 100 g edible portion (sodium in mg, everything else in g)","sources":[{"db":"USDA FDC","dataset":"SR Legacy","release":"2018-04 (final SR release; FDC CSV 2018-04)","url":"https://fdc.nal.usda.gov/download-datasets","file":"FoodData_Central_sr_legacy_food_csv_2018-04.zip","retrieved_via":"bulk CSV FoodData_Central_sr_legacy_food_csv_2018-04.zip","licence":"Public domain / CC0 1.0 (USDA ARS). Suggested citation: U.S. Department of Agriculture, Agricultural Research Service. FoodData Central, SR Legacy (2018). fdc.nal.usda.gov"},{"db":"USDA FDC","dataset":"Foundation Foods","release":"2026-04-30","url":"https://fdc.nal.usda.gov/download-datasets","file":"FoodData_Central_foundation_food_csv_2026-04-30.zip","retrieved_via":"bulk CSV FoodData_Central_foundation_food_csv_2026-04-30.zip","licence":"Public domain / CC0 1.0 (USDA ARS). Suggested citation: U.S. Department of Agriculture, Agricultural Research Service. FoodData Central, Foundation Foods (2026-04-30). fdc.nal.usda.gov"},{"db":"TFND","dataset":"食品營養成分資料集 (Taiwan Food Nutrient Database, open-data export)","release":"data.gov.tw metadata updated 2026-08-27 14:08; update frequency 每3月","url":"https://data.gov.tw/dataset/8543","file":"https://data.fda.gov.tw/data/opendata/export/20/csv (zip, UTF-8 CSV)","publisher":"衛生福利部食品藥物管理署 (Taiwan FDA)","licence":"政府資料開放授權條款-第1版 (Open Government Data License, version 1.0); attribution required","attribution":"資料來源：衛生福利部食品藥物管理署「食品營養成分資料庫」，政府資料開放平臺 data.gov.tw 資料集 8543，依政府資料開放授權條款第1版釋出。"}],"nutrients":{"water":{"unit":"g","usda_nutrient_id":[1051],"tfnd_item":"水分"},"protein":{"unit":"g","usda_nutrient_id":[1003],"tfnd_item":"粗蛋白"},"fat":{"unit":"g","usda_nutrient_id":[1004],"tfnd_item":"粗脂肪"},"ash":{"unit":"g","usda_nutrient_id":[1007],"tfnd_item":"灰分"},"carbohydrate":{"unit":"g","usda_nutrient_id":[1005],"tfnd_item":"總碳水化合物"},"sugars_total":{"unit":"g","usda_nutrient_id":[2000,1063],"tfnd_item":"糖質總量"},"sodium":{"unit":"mg","usda_nutrient_id":[1093],"tfnd_item":"鈉"}},"fields":{"n":"USDA data_points / TFND 樣本數","sd":"TFND 標準差 only","min/max/median":"USDA only, when reported","derivation":"USDA derivation code: description","n_factor":"nitrogen-to-protein factor (USDA food_protein_conversion_factor); null = not given by the database"},"notes":["All values are copied from the official releases by build_reference_foods.py; no value is typed by hand.","TFND 粗蛋白 open data does not state the nitrogen-to-protein factor, so n_factor is null for TFND entries.","USDA SR Legacy 'Sugars, Total' can be derived from other nutrients (see derivation) rather than measured.","'match' = exact | closest; read match_note and the food's variability_note before interpreting a comparison."],"subset_of":"data/reference_foods.json","subset_keys":["brown_sugar","white_sugar","brown_rice","white_rice","whole_milk_powder","soy_sauce","soy_sauce_low_sodium","potato_chips","soup_prepared","soup_powder_dry","sports_drink"]},"foods":[{"key":"whole_milk_powder","name_zh":"全脂奶粉","name_en":"Whole milk powder","tags":["dairy_powder"],"variability_note":"奶粉脂肪/蛋白質隨品牌與標準化程度而異；課程盲樣是單一品牌，不一定等於資料庫平均。","entries":[{"db":"USDA","dataset":"SR Legacy","id":170876,"name":"Milk, dry, whole, with added vitamin D","n_factor":6.38,"nutrients":{"water":{"value":2.47,"unit":"g","n":25,"nutrient_id":1051},"protein":{"value":26.32,"unit":"g","n":22,"nutrient_id":1003},"fat":{"value":26.71,"unit":"g","n":30,"nutrient_id":1004},"ash":{"value":6.08,"unit":"g","n":20,"nutrient_id":1007},"carbohydrate":{"value":38.42,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":38.42,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sodium":{"value":371.0,"unit":"mg","n":7,"nutrient_id":1093}},"match":"exact","match_note":"課程指定主要參考值。注意：170876 的正式名稱是 'with added vitamin D'；'without added vitamin D' 是 173454，兩者一般成分數值相同。"},{"db":"USDA","dataset":"SR Legacy","id":173454,"name":"Milk, dry, whole, without added vitamin D","n_factor":6.38,"nutrients":{"water":{"value":2.47,"unit":"g","n":25,"nutrient_id":1051},"protein":{"value":26.32,"unit":"g","n":22,"nutrient_id":1003},"fat":{"value":26.71,"unit":"g","n":30,"nutrient_id":1004},"ash":{"value":6.08,"unit":"g","n":20,"nutrient_id":1007},"carbohydrate":{"value":38.42,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":38.42,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sodium":{"value":371.0,"unit":"mg","n":7,"nutrient_id":1093}},"match":"exact","match_note":"170876 的不添加維生素 D 版本（一般成分相同）。"},{"db":"TFND","dataset":"食品營養成分資料集","id":"L0200101","name":"全脂奶粉","name_en":"Whole milk powder ","description":"樣品狀態:全脂(26-42%)奶粉; 前處理描述:混合均勻","category":"乳品類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":2.8,"unit":"g","n":2,"sd":0.1,"tfnd_item":"水分"},"protein":{"value":26.4,"unit":"g","n":2,"sd":0.3,"tfnd_item":"粗蛋白"},"fat":{"value":28.2,"unit":"g","n":2,"sd":0.6,"tfnd_item":"粗脂肪"},"ash":{"value":5.6,"unit":"g","n":2,"sd":0.0,"tfnd_item":"灰分"},"carbohydrate":{"value":37.0,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":36.4,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":337.0,"unit":"mg","n":2,"sd":70.0,"tfnd_item":"鈉"}},"match":"exact"}]},{"key":"soy_sauce","name_zh":"醬油","name_en":"Soy sauce (brewed, soy + wheat)","tags":["liquid","salty"],"variability_note":"醬油鈉含量品牌差異很大（一般、淡色、薄鹽/低鈉可差 30–50%）；資料庫值只代表被分析的那幾個產品。液體樣品的量測結果若是 mg/100 mL，要乘上密度換算成 mg/100 g 才能跟資料庫比。","entries":[{"db":"USDA","dataset":"SR Legacy","id":174277,"name":"Soy sauce made from soy and wheat (shoyu)","n_factor":6.25,"nutrients":{"water":{"value":71.15,"unit":"g","n":6,"min":71.05,"max":71.26,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":8.14,"unit":"g","n":6,"min":8.05,"max":8.22,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":0.57,"unit":"g","n":6,"min":0.44,"max":0.69,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":15.21,"unit":"g","n":6,"min":15.17,"max":15.28,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":4.93,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":0.4,"unit":"g","n":0,"nutrient_id":2000},"sodium":{"value":5493.0,"unit":"mg","n":6,"min":5390.0,"max":5660.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"exact","match_note":"shoyu, made from soy and wheat"},{"db":"TFND","dataset":"食品營養成分資料集","id":"P0700101","name":"醬油","name_en":"Soy sauce","description":"前處理描述:混合均勻(黃豆,小麥,鹽,糖,調味料等)","category":"調味料及香辛料類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":67.0,"unit":"g","n":2,"sd":2.6,"tfnd_item":"水分"},"protein":{"value":7.8,"unit":"g","n":2,"sd":0.0,"tfnd_item":"粗蛋白"},"ash":{"value":10.4,"unit":"g","n":2,"sd":2.9,"tfnd_item":"灰分"},"carbohydrate":{"value":14.7,"unit":"g","tfnd_item":"總碳水化合物"},"sodium":{"value":4997.0,"unit":"mg","n":2,"sd":123.0,"tfnd_item":"鈉"}},"match":"exact"}]},{"key":"soy_sauce_low_sodium","name_zh":"薄鹽醬油（低鈉）","name_en":"Soy sauce, low/reduced sodium","tags":["liquid","salty"],"variability_note":"低鈉產品的配方（以鉀鹽取代等）差異很大。","entries":[{"db":"USDA","dataset":"SR Legacy","id":172473,"name":"Soy sauce made from soy and wheat (shoyu), low sodium","n_factor":6.25,"nutrients":{"water":{"value":75.14,"unit":"g","n":6,"min":74.96,"max":75.33,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":9.05,"unit":"g","n":6,"min":8.56,"max":9.44,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":0.3,"unit":"g","n":6,"min":0.2,"max":0.36,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":9.92,"unit":"g","n":6,"min":9.86,"max":9.96,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":5.59,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":0.5,"unit":"g","n":3,"min":0.5,"max":0.5,"nutrient_id":2000,"derivation":"A: Analytical"},"sodium":{"value":3598.0,"unit":"mg","n":6,"min":3570.0,"max":3630.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"exact","match_note":"shoyu, low sodium"},{"db":"TFND","dataset":"食品營養成分資料集","id":"P0700501","name":"薄鹽醬油(低鈉)","name_en":"Soy sauce","description":"前處理描述:混合均勻(黃豆,小麥,鹽 ,糖,酒料等)","category":"調味料及香辛料類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":71.4,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":7.4,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"ash":{"value":9.2,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":11.9,"unit":"g","tfnd_item":"總碳水化合物"},"sodium":{"value":3508.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"exact"}]},{"key":"potato_chips","name_zh":"洋芋片（原味）","name_en":"Potato chips, plain, salted","tags":["salty","high_fat"],"variability_note":"調味粉、油炸/烘焙、品牌都會影響鈉與脂肪。USDA 本身 17 個樣品的鈉就從 235 到 633 mg/100 g。","entries":[{"db":"USDA","dataset":"SR Legacy","id":169677,"name":"Snacks, potato chips, plain, salted","n_factor":6.25,"nutrients":{"water":{"value":1.86,"unit":"g","n":17,"min":1.24,"max":2.44,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":6.39,"unit":"g","n":17,"min":5.44,"max":7.19,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":33.98,"unit":"g","n":17,"min":30.74,"max":38.39,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":3.94,"unit":"g","n":17,"min":3.13,"max":4.23,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":53.83,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":0.33,"unit":"g","n":9,"min":0.3,"max":0.4,"nutrient_id":2000,"derivation":"A: Analytical"},"sodium":{"value":527.0,"unit":"mg","n":17,"min":235.0,"max":633.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"exact","match_note":"plain, salted"},{"db":"TFND","dataset":"食品營養成分資料集","id":"Q4200201","name":"洋芋片","name_en":"Potato chips","description":"前處理描述:混合均勻打碎","category":"糕餅點心類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":1.9,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":5.2,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":34.3,"unit":"g","n":2,"sd":6.0,"tfnd_item":"粗脂肪"},"ash":{"value":2.8,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":55.8,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":1.9,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":383.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"closest","match_note":"TFND 未註明口味"}]},{"key":"soup_powder_dry","name_zh":"湯粉/高湯粉（乾粉）","name_en":"Bouillon / instant soup powder, dry","tags":["salty","dry"],"variability_note":"乾粉與『依標示沖泡後』的湯差了約 20–60 倍（取決於沖泡比例），比較前一定要確認基準（as sold vs as prepared）。","entries":[{"db":"USDA","dataset":"SR Legacy","id":171562,"name":"Soup, chicken broth or bouillon, dry","n_factor":6.25,"nutrients":{"water":{"value":2.27,"unit":"g","n":4,"nutrient_id":1051},"protein":{"value":16.66,"unit":"g","n":4,"nutrient_id":1003},"fat":{"value":13.88,"unit":"g","n":4,"nutrient_id":1004},"ash":{"value":49.18,"unit":"g","n":4,"nutrient_id":1007},"carbohydrate":{"value":18.01,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":17.36,"unit":"g","n":0,"nutrient_id":2000,"derivation":"FLA: Estimated formulation based on ingredient list; Linear program used to estimate ingredients; Analytical data"},"sodium":{"value":23875.0,"unit":"mg","n":4,"nutrient_id":1093,"derivation":"LC: Label claim (back calculated from label by NDL staff; Calculated from label claim/serving (g or %RDI)"}},"match":"closest","match_note":"chicken broth or bouillon, dry"},{"db":"USDA","dataset":"SR Legacy","id":171560,"name":"Soup, beef broth or bouillon, powder, dry","n_factor":6.25,"nutrients":{"water":{"value":3.27,"unit":"g","n":3,"nutrient_id":1051},"protein":{"value":15.97,"unit":"g","n":3,"nutrient_id":1003},"fat":{"value":8.89,"unit":"g","n":3,"nutrient_id":1004},"ash":{"value":54.5,"unit":"g","n":0,"nutrient_id":1007,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"carbohydrate":{"value":17.4,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sugars_total":{"value":16.71,"unit":"g","n":0,"nutrient_id":2000,"derivation":"FLA: Estimated formulation based on ingredient list; Linear program used to estimate ingredients; Analytical data"},"sodium":{"value":26000.0,"unit":"mg","n":1,"nutrient_id":1093,"derivation":"LC: Label claim (back calculated from label by NDL staff; Calculated from label claim/serving (g or %RDI)"}},"match":"closest","match_note":"beef broth or bouillon, powder, dry"},{"db":"TFND","dataset":"食品營養成分資料集","id":"P0500201","name":"高湯塊(雞肉)","name_en":"nan","description":"前處理描述:混合均勻打碎(雞肉粉,雞油,濃縮蛋白質,鹽,糖,味素,植物油,香辛料等)","category":"調味料及香辛料類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":2.1,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":14.7,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":24.8,"unit":"g","n":1,"tfnd_item":"粗脂肪"},"ash":{"value":43.9,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":14.5,"unit":"g","tfnd_item":"總碳水化合物"},"sodium":{"value":16862.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"closest","match_note":"高湯塊（雞肉）"},{"db":"TFND","dataset":"食品營養成分資料集","id":"R8100101","name":"沖泡濃湯(海鮮)","name_en":"nan","description":"樣品狀態:鋁袋包裝; 前處理描述:混合均勻磨碎","category":"加工調理食品及其他類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":3.4,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":11.2,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":6.9,"unit":"g","n":2,"sd":3.3,"tfnd_item":"粗脂肪"},"ash":{"value":16.5,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":62.0,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":11.6,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":6118.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"closest","match_note":"沖泡濃湯（海鮮），鋁袋包裝乾粉"}]},{"key":"soup_prepared","name_zh":"沖泡後的湯（依標示沖泡）","name_en":"Soup from dry mix, prepared with water","tags":["liquid","salty","as_prepared"],"variability_note":"USDA 172924 的數值是『計算值』(n=0，由乾粉＋水的比例推算)，不是直接分析值；TFND 沒有對應的沖泡後湯品。","entries":[{"db":"USDA","dataset":"SR Legacy","id":172924,"name":"Soup, chicken broth or bouillon, dry, prepared with water","n_factor":6.25,"nutrients":{"water":{"value":98.28,"unit":"g","n":0,"nutrient_id":1051,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"protein":{"value":0.28,"unit":"g","n":0,"nutrient_id":1003,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"fat":{"value":0.23,"unit":"g","n":0,"nutrient_id":1004,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"ash":{"value":0.92,"unit":"g","n":0,"nutrient_id":1007,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"carbohydrate":{"value":0.3,"unit":"g","n":0,"nutrient_id":1005,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"sugars_total":{"value":0.29,"unit":"g","n":0,"nutrient_id":2000,"derivation":"RPI: Recipe;Per package directions;No adjustments applied, combination of source codes which incl codes other than 1,12,or 6"},"sodium":{"value":401.0,"unit":"mg","n":0,"nutrient_id":1093,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"}},"match":"closest","match_note":"chicken broth or bouillon, dry, prepared with water (calculated, n=0)"}]},{"key":"sports_drink","name_zh":"運動飲料","name_en":"Sports drink, ready-to-drink","tags":["liquid"],"variability_note":"不同品牌運動飲料鈉含量可差 2 倍以上；資料庫收錄的是特定品牌。","entries":[{"db":"USDA","dataset":"SR Legacy","id":173660,"name":"Beverages, PEPSICO QUAKER, Gatorade, G performance O 2, ready-to-drink.","n_factor":6.25,"nutrients":{"water":{"value":93.38,"unit":"g","n":1,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":0.0,"unit":"g","n":1,"nutrient_id":1003,"derivation":"LC: Label claim (back calculated from label by NDL staff; Calculated from label claim/serving (g or %RDI)"},"fat":{"value":0.0,"unit":"g","n":0,"nutrient_id":1004,"derivation":"Z: Assumed zero (Insignificant amount or not naturally occurring in a food, such as fiber in meat)"},"ash":{"value":0.14,"unit":"g","n":2,"min":0.07,"max":0.2,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":6.43,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":5.24,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NC: Calculated"},"sodium":{"value":39.0,"unit":"mg","n":2,"min":39.0,"max":39.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"closest","match_note":"Gatorade G performance O 2, ready-to-drink"},{"db":"USDA","dataset":"SR Legacy","id":175108,"name":"Beverages, COCA-COLA, POWERADE, lemon-lime flavored, ready-to-drink","n_factor":6.25,"nutrients":{"water":{"value":91.91,"unit":"g","n":1,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":0.0,"unit":"g","n":3,"min":0.0,"max":0.0,"nutrient_id":1003,"derivation":"LC: Label claim (back calculated from label by NDL staff; Calculated from label claim/serving (g or %RDI)"},"fat":{"value":0.05,"unit":"g","n":0,"nutrient_id":1004,"derivation":"FLA: Estimated formulation based on ingredient list; Linear program used to estimate ingredients; Analytical data"},"ash":{"value":0.2,"unit":"g","n":1,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":7.84,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":6.11,"unit":"g","n":3,"min":6.02,"max":6.16,"nutrient_id":2000,"derivation":"LC: Label claim (back calculated from label by NDL staff; Calculated from label claim/serving (g or %RDI)"},"sodium":{"value":42.0,"unit":"mg","n":0,"nutrient_id":1093,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"}},"match":"closest","match_note":"POWERADE lemon-lime, ready-to-drink"},{"db":"TFND","dataset":"食品營養成分資料集","id":"O0703301","name":"運動飲料","name_en":"Isotonic drink","description":"樣品狀態:瓶裝(含維生素C及礦物質Na,K,Ca,Mg等); 前處理描述:混合均勻","category":"飲料類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":92.7,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":0.1,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"ash":{"value":0.2,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":7.0,"unit":"g","tfnd_item":"總碳水化合物"},"sodium":{"value":41.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"exact"}]},{"key":"brown_sugar","name_zh":"黑糖","name_en":"Brown sugar / non-centrifugal cane sugar","tags":["sugar"],"variability_note":"USDA 'Sugars, brown' 是『精製糖＋糖蜜』的美式紅糖，不是台灣的黑糖（含蜜糖、非分蜜糖），灰分與礦物質通常低很多；TFND 黑砂糖較接近本課樣品。兩個資料庫在這裡代表不同的產品。","entries":[{"db":"USDA","dataset":"SR Legacy","id":168833,"name":"Sugars, brown","n_factor":6.25,"nutrients":{"water":{"value":1.34,"unit":"g","n":15,"min":1.1,"max":2.13,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":0.12,"unit":"g","n":15,"min":0.09,"max":0.25,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":0.0,"unit":"g","n":3,"min":0.0,"max":0.0,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":0.45,"unit":"g","n":15,"min":0.04,"max":0.87,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":98.09,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":97.02,"unit":"g","n":15,"min":92.42,"max":99.85,"nutrient_id":2000,"derivation":"A: Analytical"},"sodium":{"value":28.0,"unit":"mg","n":15,"min":3.0,"max":63.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"closest","match_note":"US brown sugar (refined sugar + molasses), not Taiwanese 黑糖"},{"db":"TFND","dataset":"食品營養成分資料集","id":"N0100401","name":"黑砂糖","name_en":"Black sugar; Brown sugar","description":"前處理描述:混合均勻","category":"糖類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":2.4,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":0.5,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":0.7,"unit":"g","n":1,"tfnd_item":"粗脂肪"},"ash":{"value":1.1,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":95.3,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":86.1,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":10.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"closest","match_note":"黑砂糖"}]},{"key":"white_sugar","name_zh":"白砂糖","name_en":"White (granulated) sugar","tags":["sugar"],"variability_note":"白砂糖接近純蔗糖，灰分極低（接近秤量極限），相對誤差會很大。","entries":[{"db":"USDA","dataset":"SR Legacy","id":169655,"name":"Sugars, granulated","n_factor":6.25,"nutrients":{"water":{"value":0.02,"unit":"g","n":8,"min":0.01,"max":0.06,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":0.0,"unit":"g","n":8,"min":0.0,"max":0.0,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":0.0,"unit":"g","n":1,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":0.01,"unit":"g","n":0,"nutrient_id":1007,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"carbohydrate":{"value":99.98,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":99.8,"unit":"g","n":7,"min":99.17,"max":100.0,"nutrient_id":2000,"derivation":"A: Analytical"},"sodium":{"value":1.0,"unit":"mg","n":8,"min":1.0,"max":1.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"exact","match_note":"granulated"},{"db":"USDA","dataset":"Foundation","id":746784,"name":"Sugars, granulated","n_factor":6.25,"nutrients":{"water":{"value":0.02,"unit":"g","n":8,"min":0.01,"max":0.06,"median":0.01,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":0.0,"unit":"g","min":0.0,"max":0.0,"median":0.0,"nutrient_id":1003,"derivation":"NC: Calculated"},"fat":{"value":0.32,"unit":"g","n":8,"min":0.11,"max":0.55,"median":0.32,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":0.07,"unit":"g","n":8,"min":0.03,"max":0.13,"median":0.05,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":99.6,"unit":"g","nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":99.8,"unit":"g","nutrient_id":1063,"derivation":"AS: Summed"},"sodium":{"value":1.0,"unit":"mg","n":8,"min":1.0,"max":1.0,"median":1.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"exact","match_note":"granulated"},{"db":"TFND","dataset":"食品營養成分資料集","id":"N0100101","name":"方糖","name_en":"Cube sugar","description":"前處理描述:混合均勻打碎(純糖)","category":"糖類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":0.4,"unit":"g","n":1,"tfnd_item":"水分"},"ash":{"value":0.0,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":99.6,"unit":"g","tfnd_item":"總碳水化合物"},"sodium":{"value":2.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"closest","match_note":"TFND 無『白砂糖』品項；方糖（純糖）為最接近者"}]},{"key":"brown_rice","name_zh":"糙米（生）","name_en":"Brown rice, raw","tags":["cereal"],"variability_note":"秈/稉米品種、產地影響灰分與蛋白質；USDA 為長粒米，TFND 為稉米平均值。","entries":[{"db":"USDA","dataset":"SR Legacy","id":169703,"name":"Rice, brown, long-grain, raw (Includes foods for USDA's Food Distribution Program)","n_factor":5.95,"nutrients":{"water":{"value":11.8,"unit":"g","n":6,"min":11.1,"max":12.3,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":7.54,"unit":"g","n":6,"min":7.21,"max":8.01,"nutrient_id":1003,"derivation":"A: Analytical"},"fat":{"value":3.2,"unit":"g","n":6,"min":3.0,"max":3.5,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":1.21,"unit":"g","n":6,"min":1.13,"max":1.32,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":76.25,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":0.66,"unit":"g","n":3,"min":0.6,"max":0.7,"nutrient_id":2000,"derivation":"A: Analytical"},"sodium":{"value":5.0,"unit":"mg","n":6,"min":5.0,"max":5.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"closest","match_note":"long-grain, raw"},{"db":"USDA","dataset":"Foundation","id":2512380,"name":"Rice, brown, long grain, unenriched, raw","n_factor":5.95,"nutrients":{"water":{"value":11.47,"unit":"g","n":8,"min":10.79,"max":11.83,"median":11.54,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":7.253,"unit":"g","min":6.6045,"max":7.9135,"median":7.1697,"nutrient_id":1003,"derivation":"NC: Calculated"},"fat":{"value":3.306,"unit":"g","n":8,"min":2.57,"max":3.73,"median":3.48,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":1.283,"unit":"g","n":8,"min":1.2,"max":1.46,"median":1.255,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":76.688,"unit":"g","nutrient_id":1005,"derivation":"NC: Calculated"},"sodium":{"value":0.0,"unit":"mg","n":8,"min":0.0,"max":0.0,"median":0.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"closest","match_note":"long grain, unenriched, raw"},{"db":"TFND","dataset":"食品營養成分資料集","id":"A05013","name":"糙稉米平均值","name_en":"Brown rice","description":"樣品狀態:生; 前處理描述:混合均勻磨碎","category":"穀物類","record_type":"樣品平均值","n_factor":null,"nutrients":{"water":{"value":12.5,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":7.4,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":2.3,"unit":"g","n":1,"tfnd_item":"粗脂肪"},"ash":{"value":1.1,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":76.6,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":0.5,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":3.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"exact","match_note":"糙稉米平均值"}]},{"key":"white_rice","name_zh":"白米（生）","name_en":"White (polished) rice, raw","tags":["cereal"],"variability_note":"USDA 很多白米品項有營養強化 (enriched)，這裡刻意選 unenriched；TFND 為稉米平均值。","entries":[{"db":"USDA","dataset":"SR Legacy","id":169756,"name":"Rice, white, long-grain, regular, raw, unenriched","n_factor":5.95,"nutrients":{"water":{"value":11.62,"unit":"g","n":58,"nutrient_id":1051},"protein":{"value":7.13,"unit":"g","n":70,"nutrient_id":1003},"fat":{"value":0.66,"unit":"g","n":23,"nutrient_id":1004},"ash":{"value":0.64,"unit":"g","n":60,"nutrient_id":1007},"carbohydrate":{"value":79.95,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":0.12,"unit":"g","n":0,"nutrient_id":2000,"derivation":"BFZN: Based on another form of the food or similar food; Concentration adjustment; No adjustment; Retention factors not used"},"sodium":{"value":5.0,"unit":"mg","n":10,"nutrient_id":1093}},"match":"closest","match_note":"long-grain, regular, raw, unenriched"},{"db":"USDA","dataset":"Foundation","id":2512381,"name":"Rice, white, long grain, unenriched, raw","n_factor":5.95,"nutrients":{"water":{"value":11.19,"unit":"g","n":8,"min":10.28,"max":12.16,"median":11.33,"nutrient_id":1051,"derivation":"A: Analytical"},"protein":{"value":7.0389,"unit":"g","min":5.8905,"max":8.33,"median":6.7533,"nutrient_id":1003,"derivation":"NC: Calculated"},"fat":{"value":1.033,"unit":"g","n":8,"min":0.72,"max":1.45,"median":1.033,"nutrient_id":1004,"derivation":"A: Analytical"},"ash":{"value":0.425,"unit":"g","n":8,"min":0.2,"max":0.61,"median":0.385,"nutrient_id":1007,"derivation":"A: Analytical"},"carbohydrate":{"value":80.3131,"unit":"g","nutrient_id":1005,"derivation":"NC: Calculated"},"sodium":{"value":0.4625,"unit":"mg","n":8,"min":0.0,"max":3.7,"median":0.0,"nutrient_id":1093,"derivation":"A: Analytical"}},"match":"closest","match_note":"long grain, unenriched, raw"},{"db":"TFND","dataset":"食品營養成分資料集","id":"A05002","name":"稉米平均值","name_en":"Short grain rice","description":"樣品狀態:生; 前處理描述:混合均勻磨碎","category":"穀物類","record_type":"樣品平均值","n_factor":null,"nutrients":{"water":{"value":14.1,"unit":"g","n":10,"sd":1.6,"tfnd_item":"水分"},"protein":{"value":7.0,"unit":"g","n":10,"sd":0.5,"tfnd_item":"粗蛋白"},"fat":{"value":0.7,"unit":"g","n":10,"sd":0.3,"tfnd_item":"粗脂肪"},"ash":{"value":0.4,"unit":"g","n":10,"sd":0.1,"tfnd_item":"灰分"},"carbohydrate":{"value":77.8,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":0.1,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":2.0,"unit":"mg","n":10,"sd":1.0,"tfnd_item":"鈉"}},"match":"exact","match_note":"稉米平均值"}]}]}"""

In [ ]:
# >>> DBCHECK-HELPERS-BEGIN
# ---- 資料庫比對小工具 (food-composition database check helpers) ------------
# 同一份程式碼嵌在 NB03 與 NB08；原始檔：colab_notebooks/_dbcheck_common.py
import io as _db_io
import json as _db_json
import math as _db_math
import os as _db_os
import urllib.request as _db_urlreq

import numpy as _db_np
import pandas as _db_pd
import matplotlib.pyplot as _db_plt

DB_REFERENCE_URL = ("https://raw.githubusercontent.com/Tai-ShengYeh/"
                    "food-analysis-lab/main/data/reference_foods.json")

# 教學假設 (teaching assumption)：資料庫只給單一平均值、沒有 min/max 或 SD 時，
# 用「相對容許差 rel」與「絕對下限 abs」取較大者，當作『不同產品之間』合理的差異範圍。
# 有 min/max 或 SD 時，也至少保留這個寬度（n=2 的 SD 或 SD=0 會嚴重低估產品間差異）。
DB_DEFAULT_TOL = {
    #  nutrient      (rel,  abs)   abs 的單位：g/100 g（鈉為 mg/100 g）
    "water":        (0.10, 0.3),
    "protein":      (0.10, 0.3),
    "fat":          (0.15, 0.3),
    "ash":          (0.15, 0.05),
    "carbohydrate": (0.10, 1.0),
    "sugars_total": (0.15, 0.5),
    "sodium":       (0.15, 5.0),
}
DB_UNITS = {"water": "g", "protein": "g", "fat": "g", "ash": "g",
            "carbohydrate": "g", "sugars_total": "g", "sodium": "mg"}
DB_NACL_FACTOR = 58.44 / 22.99  # 2.542


def load_reference(source=None, url=DB_REFERENCE_URL, timeout=5, fallback_json=None):
    """Load reference_foods.json. Order: `source` (path or URL) -> local
    data/reference_foods.json -> GitHub raw URL -> embedded fallback subset
    (REFERENCE_FALLBACK_JSON, so the notebook always runs offline).
    The source actually used is stored in ref["_loaded_from"].
    Set environment variable DBCHECK_OFFLINE=1 to skip the network."""
    offline = _db_os.environ.get("DBCHECK_OFFLINE", "") == "1"
    tries = []
    if source is not None:
        tries.append(("source", source))
    tries.append(("local", _db_os.path.join("data", "reference_foods.json")))
    if not offline and url:
        tries.append(("url", url))
    for kind, where in tries:
        try:
            if str(where).startswith(("http://", "https://")):
                if offline:
                    continue
                with _db_urlreq.urlopen(str(where), timeout=timeout) as r:
                    ref = _db_json.loads(r.read().decode("utf-8"))
            elif _db_os.path.exists(str(where)):
                with open(str(where), encoding="utf-8") as fh:
                    ref = _db_json.load(fh)
            else:
                continue
            ref["_loaded_from"] = f"{kind}: {where}"
            print(f"參考資料庫來源：{ref['_loaded_from']}（{len(ref['foods'])} 種食品）")
            return ref
        except Exception as e:  # noqa: BLE001
            print(f"讀取 {where} 失敗（{type(e).__name__}），改試下一個來源。")
    text = fallback_json if fallback_json is not None else globals().get("REFERENCE_FALLBACK_JSON")
    if text is None:
        raise RuntimeError("找不到 reference_foods.json，也沒有內嵌的備援資料。")
    ref = _db_json.loads(text)
    ref["_loaded_from"] = "embedded"
    print(f"參考資料庫來源：embedded（筆記本內嵌子集，{len(ref['foods'])} 種食品）")
    return ref


def db_food(ref, food_key):
    for f in ref["foods"]:
        if f["key"] == food_key:
            return f
    raise KeyError(f"reference 裡沒有 food_key={food_key!r}；可用：{[f['key'] for f in ref['foods']]}")


def _db_fmt(x, nd=4):
    if x is None or (isinstance(x, float) and _db_math.isnan(x)):
        return "NA"
    return f"{x:.{nd}g}" if abs(x) < 1000 else f"{x:.0f}"


def _db_entry_tag(e):
    ds = {"SR Legacy": "SR", "Foundation": "FF"}.get(e.get("dataset"), "")
    return f"{e['db']}{'-' + ds if ds else ''} {e['id']}"


def reference_range(ref, food_key, nutrient, verbose=False):
    """Combine all database entries for one food/nutrient into a plausible range.
    Each entry -> interval = union of (a) its statistics [min,max] or mean±2SD (n>=2)
    and (b) the teaching default tolerance value±max(rel·value, abs).
    The food's range = union of the entry intervals.  Returns a dict with
    low, high, center (mean of entry values), water center, and per-entry details."""
    food = db_food(ref, food_key)
    rel, absf = DB_DEFAULT_TOL[nutrient]
    rows = []
    for e in food["entries"]:
        v = e["nutrients"].get(nutrient)
        if v is None or v.get("value") is None:
            continue
        val = float(v["value"])
        half = max(rel * abs(val), absf)
        lo, hi = val - half, val + half
        rule = f"±max({rel:.0%}, {absf:g}) 教學預設"
        if v.get("min") is not None and v.get("max") is not None:
            lo, hi = min(lo, v["min"]), max(hi, v["max"])
            rule = f"min–max {_db_fmt(v['min'])}–{_db_fmt(v['max'])} ∪ " + rule
        elif v.get("sd") is not None and (v.get("n") or 0) >= 2:
            lo, hi = min(lo, val - 2 * v["sd"]), max(hi, val + 2 * v["sd"])
            rule = f"mean±2SD (SD={_db_fmt(v['sd'])}, n={v['n']}) ∪ " + rule
        lo = max(lo, 0.0)
        rows.append({"entry": _db_entry_tag(e), "db": e["db"], "dataset": e.get("dataset"),
                     "id": e["id"], "name": e["name"], "match": e.get("match"),
                     "value": val, "n": v.get("n"), "sd": v.get("sd"),
                     "min": v.get("min"), "max": v.get("max"),
                     "derivation": v.get("derivation"), "n_factor": e.get("n_factor"),
                     "low": lo, "high": hi, "rule": rule})
    if not rows:
        raise ValueError(f"{food_key} 沒有任何資料庫提供 {nutrient}")
    water = [e["nutrients"]["water"]["value"] for e in food["entries"]
             if e["nutrients"].get("water", {}).get("value") is not None]
    out = {"food_key": food_key, "nutrient": nutrient, "unit": DB_UNITS[nutrient],
           "low": min(r["low"] for r in rows), "high": max(r["high"] for r in rows),
           "center": float(_db_np.mean([r["value"] for r in rows])),
           "water_center": float(_db_np.mean(water)) if water else None,
           "entries": rows, "food": food}
    if verbose:
        print(f"{food['name_zh']} ({food_key}) — {nutrient}, {DB_UNITS[nutrient]}/100 g")
        for r in rows:
            print(f"  {r['entry']:<16} {r['name'][:48]:<48} value={_db_fmt(r['value'])}"
                  f"  n={r['n']}  -> {_db_fmt(r['low'])}–{_db_fmt(r['high'])}  [{r['rule']}]")
        print(f"  => 合理範圍 (union) = {_db_fmt(out['low'])}–{_db_fmt(out['high'])} {DB_UNITS[nutrient]}/100 g")
    return out


def _db_to_per100g(nutrient, value, unit, density=None):
    """Convert a reported value to the database basis (g or mg per 100 g). Returns (value, note)."""
    u = (unit or "").replace(" ", "").lower()
    if nutrient == "sodium":
        if u in ("", "mg/100g", "mg"):
            return value, None
        if u in ("mg/kg", "ppm", "μg/g", "ug/g"):
            return value / 10.0, "mg/kg (ppm) ÷ 10 → mg/100 g"
        if u in ("g/100g",):
            return value * 1000.0, "g/100 g × 1000 → mg/100 g"
        if u in ("mg/100ml",):
            d = density if density else 1.0
            note = (f"mg/100 mL ÷ 密度 {d:g} g/mL → mg/100 g" if density
                    else "mg/100 mL 當作 mg/100 g（未提供密度，假設 1 g/mL）")
            return value / d, note
        if u in ("mg/l",):
            d = density if density else 1.0
            return value / 10.0 / d, f"mg/L ÷ 10 ÷ 密度 {d:g} → mg/100 g"
        raise ValueError(f"不認得的鈉單位：{unit}")
    if u in ("", "%", "g/100g", "g", "%w/w"):
        return value, None
    if u in ("g/kg",):
        return value / 10.0, "g/kg ÷ 10 → g/100 g"
    if u in ("g/100ml",):
        d = density if density else 1.0
        return value / d, f"g/100 mL ÷ 密度 {d:g} → g/100 g"
    raise ValueError(f"不認得的單位：{unit}")


def check_measurement(ref, food_key, nutrient, value, unit=None, basis="wet", method=None,
                      n_factor=None, sd=None, n=None, dilution=None, density=None,
                      water=None, label=None):
    """Compare one (mean) measurement with the reference range and run diagnostic flags.
    basis: "wet" (as is) or "dry" (reported on dry matter; converted with `water`
    or the database water).  method: free text, e.g. "soxhlet", "rose_gottlieb",
    "kjeldahl", "ise", "phenol_sulfuric", "difference".  n_factor: the N-to-protein
    factor YOU used.  dilution: the dilution factor you used (to detect a forgotten one)."""
    rr = reference_range(ref, food_key, nutrient)
    food = rr["food"]
    flags, notes = [], []
    v, conv = _db_to_per100g(nutrient, float(value), unit, density)
    if conv:
        notes.append("單位換算：" + conv)
    if basis == "dry":
        w = water if water is not None else rr["water_center"]
        if w is not None:
            v = v * (1 - w / 100.0)
            notes.append(f"乾基→濕基：× (1 − {w:.2f}/100)")
    lo, hi, center = rr["low"], rr["high"], rr["center"]
    status = "範圍內" if lo <= v <= hi else ("偏低" if v < lo else "偏高")
    within = [f"{r['entry']} {'✓' if r['low'] <= v <= r['high'] else '✗'}" for r in rr["entries"]]
    ratio = v / center if center else _db_np.nan

    # (1) unit / factor errors: ratio ≈ 10^k, NaCl factor, forgotten dilution
    if status != "範圍內" and center and v > 0:
        cands = [(10, "10 倍（mg/kg vs mg/100 g；g/kg vs %；mg/L vs mg/100 mL）"),
                 (100, "100 倍（mg vs g/100 g？% vs mg/g？）"),
                 (1000, "1000 倍（mg vs g；μg vs mg）")]
        if nutrient == "sodium":
            cands.append((DB_NACL_FACTOR, "2.542 倍（NaCl 當量 vs Na，58.44/22.99）"))
        if dilution:
            cands.append((float(dilution), f"稀釋倍數 {dilution:g}（忘了乘或多乘了稀釋倍數？）"))
            if float(dilution) / 10 not in (1, 10, 100):
                cands.append((float(dilution) / 10, f"稀釋倍數/10 = {dilution / 10:g}（忘了 ×D 並 ÷10 換成每 100 g？）"))
        for f, why in cands:
            for direction, r in (("高", f), ("低", 1 / f)):
                if abs(_db_math.log10(ratio) - _db_math.log10(r)) < _db_math.log10(1.25):
                    flags.append(f"單位/換算錯誤？實測 ÷ 參考 ≈ {ratio:.3g}，約{direction} {why}")

    # (2) basis mismatch: dry-basis value reported as wet (or the reverse)
    wc = rr["water_center"]
    if wc is not None and wc >= 3 and center:
        ref_dry = center / (1 - wc / 100.0)
        if abs(v / ref_dry - 1) < 0.05 and abs(v / center - 1) > 0.08:
            flags.append(f"基準不一致？數值接近『乾基』參考值 {ref_dry:.3g}（= 濕基 {center:.3g} ÷ (1 − {wc:.1f}/100)）"
                         "；資料庫是『每 100 g 可食部分（濕基）』")
        if basis == "dry" and abs(float(value) / ref_dry - 1) > 0.08 and abs(float(value) / center - 1) < 0.05:
            flags.append("基準不一致？你標為乾基，但原始數值接近濕基參考值")

    # (3) method bias / method meaning
    m = (method or "").lower()
    tags = set(food.get("tags", []))
    hydrolysed = (("hydrol" in m or "水解" in m or "gottlieb" in m or "mojonnier" in m)
                  and not any(k in m for k in ("no hydrol", "without hydrol", "non-hydrol", "未水解")))
    if nutrient == "fat" and "soxhlet" in m and not hydrolysed and "dairy_powder" in tags:
        flags.append("方法偏差（預期偏低）：奶粉脂肪被蛋白質/乳糖包覆，直接 Soxhlet（未水解）萃取不完全；"
                     "乳品參考方法是 Röse–Gottlieb（氨–乙醇–乙醚，鹼水解）")
    if nutrient == "protein" and n_factor:
        for r in rr["entries"]:
            if r["n_factor"] and abs(r["n_factor"] - n_factor) > 1e-6:
                recalc = v / n_factor * r["n_factor"]
                flags.append(f"N 換算係數不同：你用 {n_factor:g}，{r['entry']} 用 {r['n_factor']:g} → "
                             f"重新換算 = {recalc:.2f} g/100 g")
                break
        else:
            if all(r["n_factor"] is None for r in rr["entries"]):
                notes.append("資料庫未提供 N 換算係數，無法重新換算")
    if nutrient == "sodium" and "ise" in m:
        notes.append("Na⁺ ISE 量的是離子活性（受離子強度/基質影響）；Na→NaCl ×2.542 假設所有鈉都來自 NaCl")
    if "phenol" in m:
        if nutrient == "sugars_total":
            flags.append("量的不是同一件事：酚–硫酸法是『以葡萄糖當量表示的總碳水化合物』（含寡醣、可水解多醣），"
                         "不是資料庫的『總糖』")
        elif nutrient == "carbohydrate":
            notes.append("酚–硫酸法（葡萄糖當量）≠ 差減法碳水化合物（100 − 其他四項，含膳食纖維與所有誤差）")
    if nutrient == "sugars_total" and "differ" in m:
        flags.append("量的不是同一件事：差減法碳水化合物 = 100 − (水分+灰分+蛋白質+脂肪)，包含所有碳水與所有誤差；"
                     "『總糖』只算單醣＋雙醣（奶粉主要是乳糖）")
    if nutrient == "carbohydrate" and "differ" in m:
        notes.append("差減法會繼承水分/灰分/蛋白質/脂肪的所有誤差：任何一項偏低 → 碳水化合物偏高")
    for r in rr["entries"]:
        if r["n"] == 0 and r["derivation"]:
            notes.append(f"{r['entry']} 的 {nutrient} 不是直接分析值（{r['derivation'][:40]}…）")
            break

    reminder = food.get("variability_note", "")
    if any(r["match"] == "closest" for r in rr["entries"]) or len(rr["entries"]) == 1:
        reminder = ("資料庫品項只是『最接近』或只有一筆，品牌/配方可能不同。" + reminder).strip()

    def _vals(db):
        return "; ".join(f"{r['id']}: {_db_fmt(r['value'])}" for r in rr["entries"] if r["db"] == db) or "—"

    return {"label": label if label is not None else "", "food_key": food_key, "name_zh": food["name_zh"],
            "nutrient": nutrient, "unit": f"{DB_UNITS[nutrient]}/100 g", "reported": float(value),
            "reported_unit": unit or "", "measured": round(v, 4), "sd": sd, "n": n,
            "USDA": _vals("USDA"), "TFND": _vals("TFND"),
            "ref_low": round(lo, 4), "ref_high": round(hi, 4), "ref_center": round(center, 4),
            "ratio_to_ref": round(ratio, 3), "status": status, "within_entries": ", ".join(within),
            "flags": " | ".join(flags), "notes": " | ".join(notes), "reminder": reminder}


def compare_table(ref, data, food_col="food_key", nutrient_col="nutrient", value_col="value",
                  unit_col="unit", **defaults):
    """Run check_measurement for every row of a tidy DataFrame.  Optional columns
    (used when present): label, sd, n, basis, method, n_factor, dilution, density, water."""
    out = []
    opt = ["label", "sd", "n", "basis", "method", "n_factor", "dilution", "density", "water"]
    for _, r in data.iterrows():
        kw = dict(defaults)
        for c in opt:
            if c in data.columns and _db_pd.notna(r[c]):
                kw[c] = r[c]
        out.append(check_measurement(ref, r[food_col], r[nutrient_col], r[value_col],
                                     unit=r[unit_col] if unit_col in data.columns else None, **kw))
    return _db_pd.DataFrame(out)


def show_flags(table):
    """Print status + flags + notes row by row (easier to read than a wide table)."""
    for _, r in table.iterrows():
        head = f"[{r['status']}] {r['label']} {r['name_zh']} {r['nutrient']}: {_db_fmt(r['measured'])} {r['unit']}"
        print(head + f"  (參考範圍 {r['ref_low']:.4g}–{r['ref_high']:.4g}；USDA {r['USDA']}；TFND {r['TFND']})")
        print(f"     逐筆是否涵蓋：{r['within_entries']}")
        if r["flags"]:
            for f in r["flags"].split(" | "):
                print("     ⚑ " + f)
        if r["notes"]:
            for f in r["notes"].split(" | "):
                print("     · " + f)


def plot_vs_reference(table, ncols=4, title=None):
    """Small multiples: grey bar = combined reference range, ^ = USDA values,
    s = TFND values, dot ± SD = your measurement (green in range, red outside)."""
    k = len(table)
    ncols = min(ncols, k)
    nrows = int(_db_np.ceil(k / ncols))
    fig, axes = _db_plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows), squeeze=False)
    for ax, (_, r) in zip(axes.flat, table.iterrows()):
        ax.bar(0, r["ref_high"] - r["ref_low"], bottom=r["ref_low"], width=0.6,
               color="lightgrey", edgecolor="grey", label="reference range")
        ys = [r["ref_low"], r["ref_high"], r["measured"]]
        for db, mk, dx, col in (("USDA", "^", -0.18, "tab:blue"), ("TFND", "s", 0.18, "tab:purple")):
            vals = [float(s.split(": ")[1]) for s in str(r[db]).split("; ") if ": " in s and s.split(": ")[1] != "NA"]
            if vals:
                ax.plot([dx] * len(vals), vals, mk, color=col, ms=7, label=db)
                ys += vals
        colr = "tab:green" if r["status"] == "範圍內" else "tab:red"
        err = r["sd"] if r["sd"] is not None and _db_pd.notna(r["sd"]) else None
        ax.errorbar(0, r["measured"], yerr=err, fmt="o", color=colr, ms=8, capsize=4, label="measured")
        if err:
            ys += [r["measured"] - err, r["measured"] + err]
        pad = 0.08 * (max(ys) - min(ys) or abs(max(ys)) or 1.0)
        ax.set_ylim(min(ys) - pad, max(ys) + pad)
        lab = str(r["label"])
        lab = lab if lab.isascii() else ""
        ax.set_title(f"{r['food_key']}\n{r['nutrient']} {lab}".strip(), fontsize=9)
        ax.set_ylabel(f"{r['nutrient']} ({r['unit']})", fontsize=8)
        ax.set_xticks([])
        ax.set_xlim(-0.6, 0.6)
    for ax in list(axes.flat)[k:]:
        ax.axis("off")
    leg = {}
    for ax in axes.flat:
        for h, l in zip(*ax.get_legend_handles_labels()):
            leg.setdefault(l, h)
    fig.legend(list(leg.values()), list(leg.keys()), loc="lower center", ncol=4, fontsize=8, frameon=False)
    if title:
        fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    _db_plt.show()
    return fig
# <<< DBCHECK-HELPERS-END

In [ ]:
db_ref = load_reference()          # 本機 data/ → GitHub → 內嵌子集（離線也能跑）
for s in db_ref["meta"]["sources"]:
    print(f"- {s['db']} | {s['dataset']} | {s['release']} | {s['licence'][:60]}")

# 範例：看「合理範圍」是怎麼算出來的（每一筆資料庫值、用了哪條規則、聯集後的範圍）
_ = reference_range(db_ref, "brown_rice", "ash", verbose=True)

### 8.1 灰分：全班平均 vs 資料庫

把第 2 節每組的灰分 % 取全班平均 ± SD，對應到資料庫的品項：黑糖→`brown_sugar`、白糖→`white_sugar`、糙米→`brown_rice`、白米→`white_rice`、奶粉盲樣→`whole_milk_powder`。

In [ ]:
DB_ASH_MAP = {"black_sugar": "brown_sugar", "white_sugar": "white_sugar", "brown_rice": "brown_rice",
              "polished_rice": "white_rice", "milk_powder": "whole_milk_powder"}

db_ash_in = ash_wide.groupby("sample")["ash_pct"].agg(value="mean", sd="std", n="count").reset_index()
db_ash_in["food_key"] = db_ash_in["sample"].map(DB_ASH_MAP)
db_ash_in["nutrient"] = "ash"
db_ash_in["unit"] = "%"
db_ash_in["label"] = db_ash_in["sample"]

db_ash_tbl = compare_table(db_ref, db_ash_in)
db_ash_tbl[["label", "measured", "sd", "USDA", "TFND", "ref_low", "ref_high", "status", "within_entries"]]

In [ ]:
show_flags(db_ash_tbl)
plot_vs_reference(db_ash_tbl, ncols=5, title="Ash: class mean +/- SD vs database range")

### 8.2 鈉：ISE 讀數 → 每 100 g 樣品 → 跟資料庫比

換算方式跟第 6 節相同（讀數 mg/L × 稀釋倍數 → 原樣品；洋芋片是 5.00 g 萃取成 100 mL）。兩個要注意的地方：

- **液體的基準**：醬油算出來是 **mg/100 mL**，資料庫是 **mg/100 g**。醬油含約 15% 的鹽，密度明顯大於 1，本例取 **1.18 g/mL（教學假設，不是資料庫值；請實際秤 10.00 mL 醬油確認）**。運動飲料與沖泡湯的密度接近 1，直接視為 mg/100 g。
- **即食湯是「沖泡後」的湯**：第 6 節用的標示是每份 250 g（沖泡後），所以要對應資料庫的 `soup_prepared`（沖泡後），不能跟乾粉比。

In [ ]:
DB_SOY_DENSITY = 1.18   # g/mL，教學假設（醬油約 1.15–1.20 g/mL）
DB_NA_MAP = {"soy_sauce": "soy_sauce", "potato_chips": "potato_chips", "instant_soup": "soup_prepared",
             "sports_drink": "sports_drink", "milk_powder": "whole_milk_powder"}


def db_na_per100(sample, reading_ppm):
    """ISE 讀數 (mg/L，上機溶液) -> 原樣品每 100 mL（液體）或每 100 g（固體）的 Na (mg)；換算同第 6 節。"""
    if sample == "potato_chips":                      # 5.00 g 萃取成 100 mL，直接上機
        return reading_ppm * 100.0 / 1000.0 * (100.0 / 5.0)
    return reading_ppm * NA_PRODUCTS[sample]["dilution"] / 1000.0 * 100.0


db_na_grp = na_df.copy()
db_na_grp["na_per100"] = [db_na_per100(s, r) for s, r in zip(db_na_grp["sample"], db_na_grp["reading_ppm"])]
db_na_in = db_na_grp.groupby("sample")["na_per100"].agg(value="mean", sd="std", n="count").reset_index()
db_is_soy = db_na_in["sample"] == "soy_sauce"
db_na_in["food_key"] = db_na_in["sample"].map(DB_NA_MAP)
db_na_in["nutrient"] = "sodium"
db_na_in["unit"] = np.where(db_is_soy, "mg/100mL", "mg/100g")
db_na_in["density"] = np.where(db_is_soy, DB_SOY_DENSITY, np.nan)
db_na_in["sd"] = np.where(db_is_soy, db_na_in["sd"] / DB_SOY_DENSITY, db_na_in["sd"])   # SD 也換成每 100 g
db_na_in["method"] = "ise"
db_na_in["label"] = db_na_in["sample"]

db_na_tbl = compare_table(db_ref, db_na_in)
db_na_tbl[["label", "reported", "reported_unit", "measured", "USDA", "TFND", "ref_low", "ref_high",
           "status", "within_entries"]]

In [ ]:
show_flags(db_na_tbl)
plot_vs_reference(db_na_tbl, ncols=5, title="Sodium: class mean +/- SD vs database range")

In [ ]:
# 基準很重要：同一包即食湯，「乾粉」與「沖泡後」差幾十倍
db_soup_dry = reference_range(db_ref, "soup_powder_dry", "sodium", verbose=True)
db_soup_meas = db_na_tbl.loc[db_na_tbl["label"] == "instant_soup", "measured"].iloc[0]
print(f"\n本班即食湯（沖泡後）Na = {db_soup_meas:.0f} mg/100 g；"
      f"乾粉資料庫平均 ≈ {db_soup_dry['center']:.0f} mg/100 g（約 {db_soup_dry['center'] / db_soup_meas:.0f} 倍）")

### 8.3 故意算錯的三個例子：看旗標怎麼抓錯

下面三列不是本班資料，而是用本班數字**故意做錯**的示範（G6–G8 為虛構組別）：

- **G6**：醬油讀數 × 稀釋倍數 50 得到的是 **mg/L**（= ppm），卻直接寫成 mg/100 g。
- **G7**：把**上機讀數**（稀釋 50 倍後的 ppm）直接當成 mg/100 g 報告。
- **G8**：糙米灰分以**乾基**計算（除以乾物重），卻當成一般（濕基）數值報告。

In [ ]:
db_soy_reading = na_df.loc[na_df["sample"] == "soy_sauce", "reading_ppm"].mean()        # 上機讀數 (mg/L)
db_rice_ash = db_ash_in.loc[db_ash_in["sample"] == "brown_rice", "value"].iloc[0]         # 濕基灰分 %
db_rice_water = reference_range(db_ref, "brown_rice", "water")["center"]                  # 資料庫水分 g/100 g

db_wrong = pd.DataFrame([
    dict(label="G6", food_key="soy_sauce", nutrient="sodium", value=db_soy_reading * 50,
         unit="mg/100g", dilution=50),
    dict(label="G7", food_key="soy_sauce", nutrient="sodium", value=db_soy_reading,
         unit="mg/100g", dilution=50),
    dict(label="G8", food_key="brown_rice", nutrient="ash", value=db_rice_ash / (1 - db_rice_water / 100),
         unit="%"),
])
db_wrong_tbl = compare_table(db_ref, db_wrong)
show_flags(db_wrong_tbl)

### 8.4 怎麼讀這些結果

- **status（範圍內/偏低/偏高）只是第一步**。「範圍內」不代表一定對（範圍是不同產品之間的差異，很寬）；「偏高/偏低」也不代表一定錯。
- **⚑ 旗標**提示「可能是自己算錯或方法造成」：比值 ≈ 10、100、1000、2.542 或稀釋倍數，多半是單位/換算問題；數值剛好等於乾基換算值，多半是基準問題。
- **within_entries** 逐筆列出每個資料庫品項是否涵蓋你的數值——如果只有其中一個資料庫撐住「範圍內」，要特別小心。
- **reminder** 提醒資料庫品項是否只是「最接近」的產品（例如 USDA 的 brown sugar 不是台灣黑糖）。
- 跟第 6 節的**營養標示**一起看：資料庫、標示、實驗值各有各的意義，報告時要寫清楚你拿哪一個當參考、為什麼。

### （選用）用 R 做同樣的比對

R 使用者可以用 `jsonlite` 讀同一份 `reference_foods.json`，照同樣規則算範圍：

```r
# install.packages("jsonlite")   # 若尚未安裝
library(jsonlite)
path <- "data/reference_foods.json"
if (!file.exists(path))
  path <- "https://raw.githubusercontent.com/Tai-ShengYeh/food-analysis-lab/main/data/reference_foods.json"
ref <- fromJSON(path, simplifyVector = FALSE)

tol  <- c(water = .10, protein = .10, fat = .15, ash = .15, carbohydrate = .10, sugars_total = .15, sodium = .15)
absf <- c(water = .3,  protein = .3,  fat = .3,  ash = .05, carbohydrate = 1,   sugars_total = .5,  sodium = 5)
nz <- function(x) if (is.null(x)) NA else x

ref_entries <- function(ref, food_key, nutrient) {
  food <- Filter(function(f) f$key == food_key, ref$foods)[[1]]
  rows <- lapply(food$entries, function(e) {
    v <- e$nutrients[[nutrient]]
    if (is.null(v)) return(NULL)
    half <- max(tol[[nutrient]] * abs(v$value), absf[[nutrient]])
    lo <- v$value - half; hi <- v$value + half
    if (!is.null(v$min) && !is.null(v$max)) { lo <- min(lo, v$min); hi <- max(hi, v$max) }
    else if (!is.null(v$sd) && nz(v$n) >= 2) { lo <- min(lo, v$value - 2 * v$sd); hi <- max(hi, v$value + 2 * v$sd) }
    data.frame(db = e$db, id = as.character(e$id), name = e$name, value = v$value,
               n = nz(v$n), low = max(lo, 0), high = hi)
  })
  do.call(rbind, rows)
}

tab <- ref_entries(ref, "whole_milk_powder", "fat")
print(tab)
measured <- 21.28                      # 例：W6 全班直接 Soxhlet 平均（g/100 g）
rng <- c(min(tab$low), max(tab$high))
status <- if (measured < rng[1]) "low" else if (measured > rng[2]) "high" else "within"
cat(sprintf("reference range %.2f-%.2f g/100 g -> %s\n", rng[1], rng[2], status))
```

> 另外，CRAN 套件 **NutrienTrackeR** 內建數個國家的食物成分資料庫（包含 USDA），可用於飲食營養計算；但它**沒有**台灣的 TFND，所以本課的比對仍以 `reference_foods.json`（同時收 USDA 與 TFND）為準。

## 9. 練習題 (Exercises)

### 問題 1：標準添加回收率判斷

某組加標前讀數 305 ppm、加標後讀數 480 ppm、加標量 200 ppm。計算回收率%，並判斷是否落在 90–110% 的可接受範圍。若不合格，可能的原因有哪些？

In [ ]:
# TODO: 請在這裡作答

### 問題 2：NaCl 當量換算

某即食湯粉以去離子水稀釋 1:20 後，Na⁺ ISE 讀數為 **420 ppm**。求原湯粉萃取液中的 Na 濃度(g/100 mL)與 NaCl 當量(g/100 mL)。

In [ ]:
# TODO: 請在這裡作答

### 問題 3：灰分能回答什麼

用第 2 節的黑糖 vs 白糖灰分數據，說明「灰分%的差異」能不能證明黑糖含有更多對健康有益的礦物質？灰分這個指標的極限在哪裡？

In [ ]:
# TODO: 請在這裡作答

### 問題 4：標示 vs 量測的落差

用第 6 節 `compare_df` 找出「量測值低於標示值」的產品，並討論可能原因（提示：不是每個差異都代表標示造假）。

In [ ]:
# TODO: 請在這裡作答

### 問題 5：非 NaCl 的鈉來源

如果一款產品標示鈉 400 mg/100g，但配方中同時含有食鹽、味精(MSG)、小蘇打(NaHCO₃)，直接把「鈉 × 2.542」當作「鹽含量」合理嗎？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 6：讀懂資料庫比對的旗標

用第 8.3 節 `db_wrong_tbl` 的三列（G6、G7、G8）：

1. 各列的 `ratio_to_ref`（實測 ÷ 參考中心值）是多少？旗標認為是哪一種錯誤？
2. 寫出 G6、G7 正確的換算式，把數值改正成 mg/100 g（醬油密度用 `DB_SOY_DENSITY`），再用 `check_measurement()` 重新比對一次。
3. G8 的 status 是「範圍內」，為什麼還是被標記？這告訴你「範圍內」的極限是什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 7：偏高/偏低一定是量錯嗎？

第 8.1、8.2 節中，黑糖灰分與運動飲料的鈉沒有落在資料庫範圍內。

1. 看 `within_entries`、`reminder` 與 `USDA`/`TFND` 兩欄：這兩個結果比較像「量測/計算錯誤」、「方法偏差」還是「樣品本來就跟資料庫品項不同」？
2. 運動飲料再跟第 6 節的**營養標示**比一次。資料庫、標示、實驗值三者的關係是什麼？你會在報告裡拿哪一個當參考值？
3. 如果要確認你的判斷，下一步會做什麼實驗或查證？

In [ ]:
# TODO: 請在這裡作答

## 10. 匯出結果 (Export)

In [ ]:
output_path = "nb03_ash_na_results.csv"
ash_summary.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")